# Understanding LoRA: The Smart Shortcut for Fine-Tuning

In the previous notebooks, we learned that full fine-tuning updates ALL parameters and is expensive. Now we learn about **LoRA** — a trick that achieves nearly the same quality at a fraction of the cost. Read [lora.md](./lora.md) first for the theory.

## What You'll Learn
- Why we need a cheaper alternative to full fine-tuning
- What matrices are (simple explanation)
- What "low-rank" means and why it matters
- How LoRA works step by step
- Hands-on code comparing LoRA vs. full fine-tuning

## Prerequisites
- Completed Notebook 01: What is Fine-Tuning?
- Completed Full Fine-Tuning notebook (recommended)
- Basic Python knowledge

In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="fine-tuning",
    notebook_name="03_lora.ipynb"
)

---
## 1️⃣ The Problem LoRA Solves

Remember from the full fine-tuning notebook:

```
  Full Fine-Tuning of a 7B model:
  
  ❌ Needs ~112 GB GPU memory
  ❌ Costs thousands of dollars
  ❌ Risk of catastrophic forgetting
  ❌ Slow training
  
  Can we do better? 🤔
```

**LoRA's big idea:** What if we DON'T update all 7 billion parameters? What if we only add and train a **tiny number of new parameters** instead?

```
  LoRA Fine-Tuning of a 7B model:
  
  ✅ Needs only ~6-10 GB GPU memory
  ✅ Costs much less
  ✅ No catastrophic forgetting (original weights frozen!)
  ✅ Fast training
  ✅ Gets ~95-99% of full fine-tuning quality
```

### 📝 The Post-It Note Analogy

Think of LoRA like adding **sticky notes to a textbook**:

```
  Full Fine-Tuning:                    LoRA:
  
  ┌──────────────────┐                ┌──────────────────┐
  │  TEXTBOOK         │                │  TEXTBOOK         │
  │                   │                │            ┌────┐ │
  │  Rewrite EVERY    │                │  Keep the  │NOTE│ │
  │  page of the      │                │  textbook  └────┘ │
  │  textbook with    │                │  exactly    ┌────┐│
  │  new information  │                │  as it is!  │NOTE││
  │                   │                │             └────┘│
  │  (expensive &     │                │  Just add   ┌────┐│
  │   time-consuming) │                │  small      │NOTE││
  │                   │                │  sticky     └────┘│
  │                   │                │  notes!           │
  └──────────────────┘                └──────────────────┘
  
  Cost: Buy a new textbook            Cost: A few sticky notes
  Risk: Lose old information          Risk: None! Original intact
```

---
## 2️⃣ Matrices: A Quick Crash Course

Before we can understand LoRA, we need to understand one thing: **matrices**. Don't panic! We'll make this really simple.

### 📊 What is a Matrix?

A matrix is just a **grid of numbers**. That's it!

```
  A matrix is like a spreadsheet:
  
  ┌──────────────────────┐
  │  0.5   0.3   0.1     │     This is a 3×3 matrix
  │  0.2   0.8   0.4     │     (3 rows, 3 columns)
  │  0.7   0.1   0.6     │     It has 9 numbers total
  └──────────────────────┘
```

In neural networks, the **weights** (parameters) are stored as matrices. Each layer of the model has a weight matrix that transforms the input.

```
  A transformer model is made of many weight matrices:
  
  ┌───────────┐   ┌───────────┐   ┌───────────┐
  │  Layer 1  │   │  Layer 2  │   │  Layer 3  │   ... (many more)
  │  Weight   │   │  Weight   │   │  Weight   │
  │  Matrix   │   │  Matrix   │   │  Matrix   │
  │           │   │           │   │           │
  │ 4096×4096 │   │ 4096×4096 │   │ 4096×4096 │
  │ =16M nums │   │ =16M nums │   │ =16M nums │
  └───────────┘   └───────────┘   └───────────┘
  
  Each matrix has millions of numbers!
  Full fine-tuning changes ALL of them.
```

### ✖️ Matrix Multiplication (Super Simple Version)

When data flows through a neural network, it gets **multiplied by weight matrices**. This is how the model transforms input into output.

```
  Input          Weight Matrix        Output
  (your data)    (model knowledge)    (transformed data)
  
  [1, 2, 3]  ×   [0.5  0.3]    =    [result1, result2]
                  [0.2  0.8]          
                  [0.7  0.1]          
  
  Think of it as: Input passes through a "filter" (the matrix)
                  that transforms it based on what the model learned.
```

In [ ]:
# 📊 Let's see matrices in action!

import numpy as np

# A weight matrix - this is what's inside a neural network
weight_matrix = np.array([
    [0.5, 0.3, 0.1],
    [0.2, 0.8, 0.4],
    [0.7, 0.1, 0.6]
])

print("A Weight Matrix (the 'brain' of one layer):")
print("─" * 40)
print(weight_matrix)
print(f"\nShape: {weight_matrix.shape} (3 rows × 3 columns)")
print(f"Total parameters: {weight_matrix.size}")

# Input data
input_data = np.array([1.0, 2.0, 3.0])
print(f"\nInput data: {input_data}")

# Matrix multiplication = how data flows through the network
output = input_data @ weight_matrix
print(f"Output after passing through matrix: {output}")
print(f"\nThe matrix TRANSFORMED the input!")

print("\n" + "=" * 50)
print("Now imagine this, but with HUGE matrices:")
print("=" * 50)
print(f"\nA real transformer layer: 4096 × 4096 = {4096*4096:,} parameters")
print(f"A 7B model has many such layers, totaling ~7,000,000,000 parameters")
print(f"\nFull fine-tuning = update ALL {7_000_000_000:,} of them")
print(f"LoRA = add and update only ~{7_000_000_000 * 0.001:,.0f} NEW parameters (0.1%)")

# Let's VISUALIZE low-rank decomposition!

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Create a large matrix (simulating a weight change matrix)
d = 64  # Pretend this is 4096 in a real model

# Create a LOW-RANK matrix (rank 4) to simulate weight changes
# This is what LoRA discovered: fine-tuning changes are low-rank!
rank = 4
A_true = np.random.randn(d, rank) * 0.1
B_true = np.random.randn(rank, d) * 0.1
delta_W = A_true @ B_true  # The actual weight change (low-rank)

# Compare sizes
full_params = d * d
lora_params = d * rank + rank * d

print("Weight Change Matrix Analysis")
print("=" * 50)
print(f"\nOriginal matrix size: {d}x{d} = {full_params:,} parameters")
print(f"LoRA decomposition:  {d}x{rank} + {rank}x{d} = {lora_params:,} parameters")
print(f"\nCompression ratio: {full_params/lora_params:.1f}x smaller!")
print(f"Parameters saved: {(1 - lora_params/full_params)*100:.1f}%")

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# 1. Full weight change matrix
ax = axes[0]
im = ax.imshow(delta_W, cmap='RdBu', vmin=-0.1, vmax=0.1)
ax.set_title(f'Full Weight Change\n({d}x{d} = {full_params:,} params)', fontweight='bold')
ax.set_xlabel('Columns')
ax.set_ylabel('Rows')

# 2. Matrix A
ax = axes[1]
ax.imshow(A_true, cmap='RdBu', vmin=-0.3, vmax=0.3, aspect='auto')
ax.set_title(f'LoRA Matrix A\n({d}x{rank} = {d*rank} params)', fontweight='bold')
ax.set_xlabel(f'{rank} columns')
ax.set_ylabel(f'{d} rows')

# 3. Matrix B
ax = axes[2]
ax.imshow(B_true, cmap='RdBu', vmin=-0.3, vmax=0.3, aspect='auto')
ax.set_title(f'LoRA Matrix B\n({rank}x{d} = {rank*d} params)', fontweight='bold')
ax.set_xlabel(f'{d} columns')
ax.set_ylabel(f'{rank} rows')

# 4. Reconstructed (A x B)
ax = axes[3]
reconstructed = A_true @ B_true
ax.imshow(reconstructed, cmap='RdBu', vmin=-0.1, vmax=0.1)
ax.set_title(f'A x B = Same Change!\n({lora_params} params total)', fontweight='bold')
ax.set_xlabel('Columns')

# Add annotations
fig.text(0.31, 0.02, 'x', fontsize=30, ha='center', fontweight='bold')
fig.text(0.54, 0.02, '=', fontsize=30, ha='center', fontweight='bold')

plt.suptitle(f'LoRA: Decompose Big Matrix into Two Small Ones ({full_params/lora_params:.0f}x compression!)',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print(f"\nInstead of storing {full_params:,} numbers, LoRA stores only {lora_params:,}!")
print(f"And the result is EXACTLY the same!")

In [ ]:
# 📊 Let's VISUALIZE low-rank decomposition!

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Create a large matrix (simulating a weight change matrix)
d = 64  # Pretend this is 4096 in a real model

# Create a LOW-RANK matrix (rank 4) to simulate weight changes
# This is what LoRA discovered: fine-tuning changes are low-rank!
rank = 4
A_true = np.random.randn(d, rank) * 0.1
B_true = np.random.randn(rank, d) * 0.1
delta_W = A_true @ B_true  # The actual weight change (low-rank)

# Compare sizes
full_params = d * d
lora_params = d * rank + rank * d

print("Weight Change Matrix Analysis")
print("=" * 50)
print(f"\nOriginal matrix size: {d}×{d} = {full_params:,} parameters")
print(f"LoRA decomposition:  {d}×{rank} + {rank}×{d} = {lora_params:,} parameters")
print(f"\nCompression ratio: {full_params/lora_params:.1f}× smaller!")
print(f"Parameters saved: {(1 - lora_params/full_params)*100:.1f}%")

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# 1. Full weight change matrix
ax = axes[0]
im = ax.imshow(delta_W, cmap='RdBu', vmin=-0.1, vmax=0.1)
ax.set_title(f'Full Weight Change\n({d}×{d} = {full_params:,} params)', fontweight='bold')
ax.set_xlabel('Columns')
ax.set_ylabel('Rows')

# 2. Matrix A
ax = axes[1]
ax.imshow(A_true, cmap='RdBu', vmin=-0.3, vmax=0.3, aspect='auto')
ax.set_title(f'LoRA Matrix A\n({d}×{rank} = {d*rank} params)', fontweight='bold')
ax.set_xlabel(f'{rank} columns')
ax.set_ylabel(f'{d} rows')

# 3. Multiplication sign
ax = axes[2]
ax.imshow(B_true, cmap='RdBu', vmin=-0.3, vmax=0.3, aspect='auto')
ax.set_title(f'LoRA Matrix B\n({rank}×{d} = {rank*d} params)', fontweight='bold')
ax.set_xlabel(f'{d} columns')
ax.set_ylabel(f'{rank} rows')

# 4. Reconstructed (A × B)
ax = axes[3]
reconstructed = A_true @ B_true
ax.imshow(reconstructed, cmap='RdBu', vmin=-0.1, vmax=0.1)
ax.set_title(f'A × B = Same Change!\n({lora_params} params total)', fontweight='bold')
ax.set_xlabel('Columns')

# Add annotations
fig.text(0.31, 0.02, '×', fontsize=30, ha='center', fontweight='bold')
fig.text(0.54, 0.02, '=', fontsize=30, ha='center', fontweight='bold')

plt.suptitle(f'LoRA: Decompose Big Matrix into Two Small Ones ({full_params/lora_params:.0f}× compression!)',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print(f"\nInstead of storing {full_params:,} numbers, LoRA stores only {lora_params:,}!")
print(f"And the result is EXACTLY the same!")

---
## 4️⃣ How LoRA Actually Works

Now let's put it all together. Here's LoRA's complete recipe:

### 🔧 Step-by-Step Process

```
  LoRA Step-by-Step:
  ═══════════════════
  
  Step 1: FREEZE the original model
  ┌─────────────────────────────────────────────────────┐
  │  Original Weight Matrix W (e.g., 4096 × 4096)      │
  │  🔒 FROZEN - Don't touch these!                     │
  │  These stay exactly as they were after pre-training  │
  └─────────────────────────────────────────────────────┘
  
  Step 2: ADD two small matrices (A and B) beside each layer
  ┌─────────────────────────┐  ┌────────┐  ┌────────────┐
  │  Original W (frozen)    │  │ A      │  │ B          │
  │  4096 × 4096            │  │ 4096×r │  │ r×4096     │
  │  🔒 16M params (frozen) │  │ 🔥 32K │  │ 🔥 32K     │
  └─────────────────────────┘  └────────┘  └────────────┘
                                     Only THESE get trained!
  
  Step 3: During inference, combine them
  ┌─────────────────────────────────────────────────────┐
  │  output = input × W  +  input × (A × B)            │
  │           ────────      ──────────────              │
  │           original      LoRA addition               │
  │           knowledge     (new task knowledge)         │
  └─────────────────────────────────────────────────────┘
  
  The 'r' in the middle is called the RANK.
  Common values: r = 4, 8, 16, 32, 64
  Smaller r = fewer parameters = cheaper, but slightly less quality
```

### 🎯 The "r" Value (Rank)

The **rank (r)** is the most important setting in LoRA. It controls the size of those small matrices:

| Rank (r) | LoRA Parameters | Quality | Analogy |
|----------|----------------|---------|--------|
| r = 4 | Very few | Good | Adding 4 sticky notes per page |
| r = 8 | Few | Very good | Adding 8 sticky notes |
| r = 16 | Moderate | Great | Adding 16 sticky notes |
| r = 64 | More | Near-perfect | Adding 64 sticky notes |
| r = 4096 | Same as full | Perfect | Rewriting the whole page (defeats the purpose!) |

In [ ]:
# Visualize HOW LoRA works inside the model

import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)

# ── Input ──
input_box = patches.FancyBboxPatch((0.5, 4), 2, 2, boxstyle='round,pad=0.2',
                                     facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(input_box)
ax.text(1.5, 5, 'Input\nx', ha='center', va='center', fontsize=12, fontweight='bold')

# ── Original Weight (frozen) ──
frozen_box = patches.FancyBboxPatch((4, 6), 3, 2.5, boxstyle='round,pad=0.2',
                                     facecolor='#FFCDD2', edgecolor='#C62828', linewidth=2)
ax.add_patch(frozen_box)
ax.text(5.5, 7.9, 'Original W', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(5.5, 7.2, '(4096 x 4096)', ha='center', va='center', fontsize=9)
ax.text(5.5, 6.5, 'FROZEN', ha='center', va='center', fontsize=11)

# ── LoRA path ──
a_box = patches.FancyBboxPatch((4, 1.5), 1.8, 2.5, boxstyle='round,pad=0.2',
                                 facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(a_box)
ax.text(4.9, 3.3, 'A', ha='center', va='center', fontsize=14, fontweight='bold', color='#2E7D32')
ax.text(4.9, 2.7, f'(4096 x r)', ha='center', va='center', fontsize=8)
ax.text(4.9, 2.1, 'train', ha='center', va='center', fontsize=9)

b_box = patches.FancyBboxPatch((6.5, 1.5), 1.8, 2.5, boxstyle='round,pad=0.2',
                                 facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(b_box)
ax.text(7.4, 3.3, 'B', ha='center', va='center', fontsize=14, fontweight='bold', color='#2E7D32')
ax.text(7.4, 2.7, f'(r x 4096)', ha='center', va='center', fontsize=8)
ax.text(7.4, 2.1, 'train', ha='center', va='center', fontsize=9)

# ── Addition node ──
circle = plt.Circle((10, 5), 0.6, facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)
ax.add_patch(circle)
ax.text(10, 5, '+', ha='center', va='center', fontsize=24, fontweight='bold', color='#F57F17')

# ── Output ──
output_box = patches.FancyBboxPatch((11.5, 4), 2, 2, boxstyle='round,pad=0.2',
                                      facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(output_box)
ax.text(12.5, 5, 'Output\nh', ha='center', va='center', fontsize=12, fontweight='bold')

# ── Arrows ──
arrow_style = dict(arrowstyle='->', color='gray', lw=2)
ax.annotate('', xy=(4, 7.25), xytext=(2.5, 5.5), arrowprops=arrow_style)
ax.annotate('', xy=(4, 2.75), xytext=(2.5, 4.5), arrowprops=arrow_style)
ax.annotate('', xy=(6.5, 2.75), xytext=(5.8, 2.75), arrowprops=arrow_style)
ax.annotate('', xy=(9.5, 5.5), xytext=(7, 7.25), arrowprops=arrow_style)
ax.annotate('', xy=(9.5, 4.5), xytext=(8.3, 2.75), arrowprops=arrow_style)
ax.annotate('', xy=(11.5, 5), xytext=(10.6, 5), arrowprops=arrow_style)

ax.text(3, 7, 'Original\npath', fontsize=9, ha='center', color='#C62828', fontstyle='italic')
ax.text(3, 2.5, 'LoRA\npath', fontsize=9, ha='center', color='#2E7D32', fontstyle='italic')

ax.set_title('How LoRA Works Inside a Neural Network Layer', 
             fontsize=16, fontweight='bold', pad=20)
ax.text(7, 0.5, 'Formula:  h = x * W  +  x * A * B', 
        ha='center', fontsize=13, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))

ax.axis('off')
plt.tight_layout()
plt.show()

print("Key idea: The original weights are FROZEN (red box).")
print("Only the small green LoRA matrices (A and B) get trained!")
print("The outputs of both paths are ADDED together.")

In [ ]:
# 📊 Visualize HOW LoRA works inside the model

import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)

# ── Input ──
input_box = patches.FancyBboxPatch((0.5, 4), 2, 2, boxstyle='round,pad=0.2',
                                     facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(input_box)
ax.text(1.5, 5, 'Input\nx', ha='center', va='center', fontsize=12, fontweight='bold')

# ── Original Weight (frozen) ──
frozen_box = patches.FancyBboxPatch((4, 6), 3, 2.5, boxstyle='round,pad=0.2',
                                     facecolor='#FFCDD2', edgecolor='#C62828', linewidth=2)
ax.add_patch(frozen_box)
ax.text(5.5, 7.9, 'Original W', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(5.5, 7.2, '(4096 × 4096)', ha='center', va='center', fontsize=9)
ax.text(5.5, 6.5, '🔒 FROZEN', ha='center', va='center', fontsize=11)

# ── LoRA path ──
# Matrix A (down projection)
a_box = patches.FancyBboxPatch((4, 1.5), 1.8, 2.5, boxstyle='round,pad=0.2',
                                 facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(a_box)
ax.text(4.9, 3.3, 'A', ha='center', va='center', fontsize=14, fontweight='bold', color='#2E7D32')
ax.text(4.9, 2.7, f'(4096×r)', ha='center', va='center', fontsize=8)
ax.text(4.9, 2.1, '🔥 train', ha='center', va='center', fontsize=9)

# Matrix B (up projection)
b_box = patches.FancyBboxPatch((6.5, 1.5), 1.8, 2.5, boxstyle='round,pad=0.2',
                                 facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(b_box)
ax.text(7.4, 3.3, 'B', ha='center', va='center', fontsize=14, fontweight='bold', color='#2E7D32')
ax.text(7.4, 2.7, f'(r×4096)', ha='center', va='center', fontsize=8)
ax.text(7.4, 2.1, '🔥 train', ha='center', va='center', fontsize=9)

# ── Addition node ──
circle = plt.Circle((10, 5), 0.6, facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)
ax.add_patch(circle)
ax.text(10, 5, '+', ha='center', va='center', fontsize=24, fontweight='bold', color='#F57F17')

# ── Output ──
output_box = patches.FancyBboxPatch((11.5, 4), 2, 2, boxstyle='round,pad=0.2',
                                      facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(output_box)
ax.text(12.5, 5, 'Output\nh', ha='center', va='center', fontsize=12, fontweight='bold')

# ── Arrows ──
arrow_style = dict(arrowstyle='->', color='gray', lw=2)

# Input → W (top path)
ax.annotate('', xy=(4, 7.25), xytext=(2.5, 5.5), arrowprops=arrow_style)
# Input → A (bottom path)  
ax.annotate('', xy=(4, 2.75), xytext=(2.5, 4.5), arrowprops=arrow_style)
# A → B
ax.annotate('', xy=(6.5, 2.75), xytext=(5.8, 2.75), arrowprops=arrow_style)
# W → +
ax.annotate('', xy=(9.5, 5.5), xytext=(7, 7.25), arrowprops=arrow_style)
# B → +
ax.annotate('', xy=(9.5, 4.5), xytext=(8.3, 2.75), arrowprops=arrow_style)
# + → Output
ax.annotate('', xy=(11.5, 5), xytext=(10.6, 5), arrowprops=arrow_style)

# ── Labels ──
ax.text(3, 7, 'Original\npath', fontsize=9, ha='center', color='#C62828', fontstyle='italic')
ax.text(3, 2.5, 'LoRA\npath', fontsize=9, ha='center', color='#2E7D32', fontstyle='italic')

# ── Title and formula ──
ax.set_title('How LoRA Works Inside a Neural Network Layer', 
             fontsize=16, fontweight='bold', pad=20)
ax.text(7, 0.5, 'Formula:  h = x × W  +  x × A × B', 
        ha='center', fontsize=13, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))

ax.axis('off')
plt.tight_layout()
plt.show()

print("Key idea: The original weights are FROZEN (red box).")
print("Only the small green LoRA matrices (A and B) get trained!")
print("The outputs of both paths are ADDED together.")

---
## 5️⃣ LoRA vs Full Fine-Tuning: Side by Side

Let's compare both approaches on the same task to see the difference.

In [ ]:
# Visualize the comparison

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Loss comparison
ax = axes[0]
ax.plot(full_losses, label='Full Fine-Tuning', color='#F44336', linewidth=2)
ax.plot(lora_losses, label='LoRA (rank=2)', color='#4CAF50', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (lower = better)')
ax.set_title('Training Loss Comparison', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Accuracy comparison
ax = axes[1]
ax.plot(full_accs, label='Full Fine-Tuning', color='#F44336', linewidth=2)
ax.plot(lora_accs, label='LoRA (rank=2)', color='#4CAF50', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (higher = better)')
ax.set_title('Training Accuracy Comparison', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0.4, 1.05)

# 3. Parameter efficiency
ax = axes[2]
methods = ['Full\nFine-Tuning', 'LoRA\n(rank=2)']
trainable = [10, rank * 10 + rank]
frozen = [0, 10]

bars1 = ax.bar(methods, trainable, color=['#F44336', '#4CAF50'], alpha=0.8,
               label='Trainable params', edgecolor='black')
bars2 = ax.bar(methods, frozen, bottom=trainable, color=['#FFCDD2', '#C8E6C9'], alpha=0.8,
               label='Frozen params', edgecolor='black')

for bar, count in zip(bars1, trainable):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
            f'{count} trainable', ha='center', va='center', fontweight='bold', fontsize=10)

ax.set_ylabel('Number of Parameters')
ax.set_title('Parameter Efficiency', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Full Fine-Tuning vs LoRA: Nearly Same Quality, Much Fewer Parameters!',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nLoRA achieves {lora_test_acc/full_test_acc*100:.0f}% of full fine-tuning's accuracy")
print(f"while training only {(rank*10+rank)/10*100:.0f}% of the parameters!")

In [ ]:
# 📊 Visualize the comparison

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Loss comparison
ax = axes[0]
ax.plot(full_losses, label='Full Fine-Tuning', color='#F44336', linewidth=2)
ax.plot(lora_losses, label='LoRA (rank=2)', color='#4CAF50', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (lower = better)')
ax.set_title('Training Loss Comparison', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Accuracy comparison
ax = axes[1]
ax.plot(full_accs, label='Full Fine-Tuning', color='#F44336', linewidth=2)
ax.plot(lora_accs, label='LoRA (rank=2)', color='#4CAF50', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (higher = better)')
ax.set_title('Training Accuracy Comparison', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0.4, 1.05)

# 3. Parameter efficiency
ax = axes[2]
methods = ['Full\nFine-Tuning', 'LoRA\n(rank=2)']
trainable = [10, rank * 10 + rank]
frozen = [0, 10]

bars1 = ax.bar(methods, trainable, color=['#F44336', '#4CAF50'], alpha=0.8,
               label='Trainable params', edgecolor='black')
bars2 = ax.bar(methods, frozen, bottom=trainable, color=['#FFCDD2', '#C8E6C9'], alpha=0.8,
               label='Frozen params', edgecolor='black')

for bar, count in zip(bars1, trainable):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
            f'{count} trainable', ha='center', va='center', fontweight='bold', fontsize=10)

ax.set_ylabel('Number of Parameters')
ax.set_title('Parameter Efficiency', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Full Fine-Tuning vs LoRA: Nearly Same Quality, Much Fewer Parameters!',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nLoRA achieves {lora_test_acc/full_test_acc*100:.0f}% of full fine-tuning's accuracy")
print(f"while training only {(rank*10+rank)/10*100:.0f}% of the parameters!")

---
## 6️⃣ LoRA in Practice: Real-World Usage

Here's how LoRA is used in the real world with popular libraries:

```python
# This is what real LoRA fine-tuning looks like (using Hugging Face PEFT library)
# DON'T run this - it's just to show you the real-world code!

from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# Step 1: Load a pre-trained model
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b")

# Step 2: Define LoRA configuration
lora_config = LoraConfig(
    r=8,                # Rank (how big are the adapter matrices)
    lora_alpha=32,      # Scaling factor
    target_modules=[    # Which layers to add LoRA to
        "q_proj",       # Query projection
        "v_proj",       # Value projection
    ],
    lora_dropout=0.05,  # Regularization
)

# Step 3: Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Step 4: Check how many parameters are trainable
model.print_trainable_parameters()
# Output: "trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.06%"
# Only 0.06% of parameters are trained! The rest are frozen.
```

### 🎛️ Common LoRA Hyperparameters

| Parameter | What It Controls | Recommended Values |
|-----------|-----------------|-------------------|
| `r` (rank) | Size of LoRA matrices | 4-64 (8 or 16 is common) |
| `lora_alpha` | How much LoRA affects the output | Usually 2×r (e.g., 16 for r=8) |
| `target_modules` | Which layers get LoRA | Usually attention layers (q, k, v, o projections) |
| `lora_dropout` | Prevents overfitting | 0.0 to 0.1 |

---
## Key Takeaways

1. **LoRA freezes the original model** and adds tiny trainable "adapter" matrices

2. **It's based on a math insight**: fine-tuning changes are low-rank and can be compressed

3. **Two small matrices (A and B)** replace one huge weight change matrix

4. **The rank (r)** controls the tradeoff: higher r = better quality but more parameters

5. **LoRA avoids catastrophic forgetting** because original weights don't change

6. **You can swap LoRA adapters** to use one model for multiple tasks

7. **In practice**, LoRA achieves 95-99% of full fine-tuning quality at ~1% of the cost

## What's Next?

Can we make LoRA even cheaper? Yes! Enter **QLoRA** — which adds quantization (compression) to make fine-tuning possible on consumer GPUs.

| Next Notebook | What You'll Learn |
|---------------|------------------|
| [QLoRA](./04_qlora.ipynb) | How quantization makes LoRA even more efficient |

For interview depth, read [lora-interview.md](./lora-interview.md).

---

[Back to Module README](./README.md)

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)

---
## Key Takeaways

1. **LoRA freezes the original model** and adds tiny trainable "adapter" matrices

2. **It's based on a math insight**: fine-tuning changes are low-rank and can be compressed

3. **Two small matrices (A and B)** replace one huge weight change matrix

4. **The rank (r)** controls the tradeoff: higher r = better quality but more parameters

5. **LoRA avoids catastrophic forgetting** because original weights don't change

6. **You can swap LoRA adapters** to use one model for multiple tasks

7. **In practice**, LoRA achieves 95-99% of full fine-tuning quality at ~1% of the cost

## What's Next?

Can we make LoRA even cheaper? Yes! Enter **QLoRA** - which adds quantization (compression) to make fine-tuning possible on consumer GPUs.

| Next Notebook | What You'll Learn |
|---------------|------------------|
| [QLoRA](./04_qlora.ipynb) | How quantization makes LoRA even more efficient |

For interview-depth coverage of LoRA math, rank selection, and failure modes, read [lora-interview.md](./lora-interview.md).

---

[Back to Module README](./README.md)